In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, json, pickle
import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

from go_ml.models.esm_token_finetune import ESMCTokenFinetune
from go_ml.data_utils import *
from argparse import ArgumentParser
import transformers


parser = ArgumentParser()
parser = ESMCTokenFinetune.add_model_specific_args(parser)
hparams = parser.parse_args()
print("got hparams", hparams, type(hparams))

import pickle
import pandas as pd
from go_ml.eval_utils import filter_annot_df
from go_ml.eval_utils import (load_msa_dict, gen_bert_mat, get_bert_entropy, 
                              gen_annot_mat, gen_seq_len_mask, mean_reciprocal_rank, 
                              mean_reciprocal_rank_mat, mean_auc, top_30_score, roc_average)

data_root = '../gen_datasets/datasets'
annot_df = filter_annot_df(pd.read_csv(f'{data_root}/csa_annot.csv', sep='\t'))
# llps_df = filter_annot_df(pd.read_csv(f'{data_root}/llps_dataset.csv', sep='\t'))
# elms_df = filter_annot_df(pd.read_csv(f'{data_root}/elms_dataset.csv', sep='\t'))

np.random.seed(42)
train_row_mask = np.random.rand(len(annot_df)) > 0.5
#select rows from b
train_df = annot_df[train_row_mask]
val_df = annot_df[~train_row_mask]

import json
train_path = "/home/andrew/cafa5_team/data/"
with open(f"{train_path}/cafa_dataset/go_terms.json", "r") as f:
    go_terms = json.load(f)

dataloader_params = {"shuffle": True, "batch_size": 3, "collate_fn": prot_func_collate}
val_dataloader_params = {"shuffle": False, "batch_size": 12, "collate_fn": prot_func_collate}

train_dataset = ProtFuncDataset.from_annot_df(train_df, go_terms)
val_dataset = ProtFuncDataset.from_annot_df(val_df, go_terms)
train_loader = DataLoader(train_dataset, **dataloader_params)
val_loader = DataLoader(val_dataset, **dataloader_params)


got hparams Namespace(model_name='Synthyra/ESMplusplus_small', max_length=1024, learning_rate=1.5e-05, gradient_checkpointing=False, freeze_encoder=True, gradient_clipping=1.0, weight_decay=0.01) <class 'argparse.Namespace'>


In [ ]:
hparams.num_train_steps = 25*len(annot_df)
model = ESMCTokenFinetune(hparams)

import torch.nn.functional as F
batch = next(iter(train_loader))
seq_ind, mask, annot_labels = (batch['seq_ind'], batch['mask'], batch['annot_mask'])
esm_output = model.forward(seq_ind, mask)
token_logits = esm_output.logits[:, :annot_labels.shape[1], :]
annot_labels = annot_labels[:, :token_logits.shape[1]].long()
annot_labels[~mask.bool()] = -100
loss_val = F.cross_entropy(token_logits.reshape(-1, 2), annot_labels.reshape(-1), ignore_index=-100)


Some weights of ESMplusplusForTokenClassification were not initialized from the model checkpoint at Synthyra/ESMplusplus_small and are newly initialized: ['classifier.0.bias', 'classifier.0.weight', 'classifier.2.bias', 'classifier.2.weight', 'classifier.3.bias', 'classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Froze ESM model parameters
torch.Size([3, 421, 2]) torch.Size([3, 421])


In [ ]:

# early_stop_callback = EarlyStopping(monitor='loss/val', min_delta=0.00, patience=3, verbose=True, mode='min')
# checkpoint_callback = ModelCheckpoint(filename="/home/andrew/GO_interp/checkpoints/esm_finetune", 
#                                         verbose=True, monitor='loss/val', mode='min')
# from pytorch_lightning.loggers import TensorBoardLogger
# logger = TensorBoardLogger("annot_logs", name="esmc_warmup", default_hp_metric=False)
# trainer = pl.Trainer(devices=[0], max_epochs=10, 
#                         callbacks=[early_stop_callback, checkpoint_callback], 
#                         logger=logger, precision="bf16-mixed")
# trainer.fit(model, train_loader, val_loader)

Some weights of ESMplusplusForTokenClassification were not initialized from the model checkpoint at Synthyra/ESMplusplus_small and are newly initialized: ['classifier.0.bias', 'classifier.0.weight', 'classifier.2.bias', 'classifier.2.weight', 'classifier.3.bias', 'classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Froze ESM model parameters


RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

In [10]:
from transformers import AutoModelForTokenClassification
from transformers import AutoTokenizer
import torch
device = torch.device('cuda:0')
model_name = 'Synthyra/ESMplusplus_small'
model = AutoModelForTokenClassification.from_pretrained(model_name, torch_dtype='auto', trust_remote_code=True).train().to(device)

Some weights of ESMplusplusForTokenClassification were not initialized from the model checkpoint at Synthyra/ESMplusplus_small and are newly initialized: ['classifier.0.bias', 'classifier.0.weight', 'classifier.2.bias', 'classifier.2.weight', 'classifier.3.bias', 'classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
for key in batch:
    if type(batch[key]) is torch.Tensor:
        batch[key] = batch[key].to(device)

In [26]:
import numpy as np
annot_df = csa_df
train_row_mask = np.random.rand(len(annot_df)) > 0.5
#select rows from b
train_df = annot_df[train_row_mask]
val_df = annot_df[~train_row_mask]

In [27]:
train_dataset = ProtFuncDataset.from_annot_df(train_df, go_terms)
val_dataset = ProtFuncDataset.from_annot_df(val_df, go_terms)
train_loader = DataLoader(train_dataset, **dataloader_params)
val_loader = DataLoader(val_dataset, **dataloader_params)

In [12]:
out = model.forward(batch['seq_ind'], attention_mask=batch['mask'])

In [16]:
import inspect
inspect.getsourcefile(model.forward)

'/home/andrew/.cache/huggingface/modules/transformers_modules/Synthyra/ESMplusplus_small/1e40c1b8ef46c33f93a9b817eb0bd81279ab4088/modeling_esm_plusplus.py'